In [1]:
# 01_synthetic_proxy_mechanism.ipynb
#
# Experiment 1 -- Controlled synthetic illustration of the proxy mechanism.
#
# Purpose:
#   Illustrate, on fully synthetic data with a NON-CIRCULAR generator, that
#   discrimination placed only in the historical labelling decision (not injected
#   through the proxies) still reaches a protected-blind model via proxies that
#   merely correlate with the protected group. Shows both axes: outcome parity
#   (DIR) and proxy leakage (reconstruction AUC). This experiment is an
#   illustration of the mechanism, not evidence of effect size; the real-data
#   experiments carry the empirical claims. 30 seeds, 95% CIs.
#
# Design (non-circular):
#   protected A; proxies region, spending correlated with A but NOT in the label;
#   legitimate features income, debt, payhist determine TRUE creditworthiness;
#   the OBSERVED label applies a historical bias to group A at the decision stage.
#
# Outputs (PNG+PDF, 600 dpi, grayscale, no captions; paths not printed):
#   data/synthetic_credit.csv  (one representative seed, for reference)
#   results/tables/exp1_metrics.csv
#   results/figures/exp1_auc_dir.(png|pdf)
#   results/figures/exp1_dir_by_condition.(png|pdf)

import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.isdir(os.path.join(PROJECT_ROOT, "results")):
    PROJECT_ROOT = os.getcwd()
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
FIG_DIR  = os.path.join(PROJECT_ROOT, "results", "figures")
TAB_DIR  = os.path.join(PROJECT_ROOT, "results", "tables")
for _d in (DATA_DIR, FIG_DIR, TAB_DIR):
    os.makedirs(_d, exist_ok=True)

sns.set_theme(style="whitegrid")
GRAYS = ["#000000", "#555555", "#999999", "#cccccc"]
sns.set_palette(sns.color_palette(GRAYS))
plt.rcParams.update({"figure.dpi": 600, "savefig.dpi": 600, "font.size": 11,
    "axes.edgecolor": "black", "axes.linewidth": 0.8, "grid.color": "0.85"})

def save_fig(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=600, bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, name + ".pdf"), dpi=600, bbox_inches="tight")

def dir_ratio(y_pred, group):
    approve = (y_pred == 0).astype(int)
    r1 = approve[group==1].mean(); r0 = approve[group==0].mean()
    return (r1/r0) if r0 > 0 else np.nan

def eo_gaps(y_true, y_pred, group):
    def rate(ct, mask):
        idx=(y_true==ct)&mask
        return (y_pred[idx]==1).mean() if idx.sum() else np.nan
    a,b = group==1, group==0
    return abs(rate(1,a)-rate(1,b)), abs(rate(0,a)-rate(0,b))

def leakage_auc(Xdf, g):
    Xs = StandardScaler().fit_transform(Xdf.values)
    try:
        return cross_val_score(LogisticRegression(max_iter=1000), Xs, g, cv=3,
                               scoring="roc_auc").mean()
    except Exception:
        return np.nan

def generate(seed, n=8000, proxy_corr=1.4, bias=0.9):
    rng = np.random.default_rng(seed)
    A = rng.binomial(1, 0.35, n)
    region   = proxy_corr*A + rng.normal(0, 1.0, n)
    spending = proxy_corr*A + rng.normal(0, 1.0, n)
    income   = rng.normal(0,1,n); debt = rng.normal(0,1,n); payhist = rng.normal(0,1,n)
    true_latent  = 0.9*income - 0.8*debt + 0.7*payhist + rng.normal(0,0.8,n)
    true_default = (true_latent < np.quantile(true_latent, 0.30)).astype(int)
    # historical bias in the LABELLING decision, applied by group membership
    flip_up = (A==1) & (true_default==0) & (rng.random(n) < bias*0.25)
    observed = np.where(flip_up, 1, true_default)
    return pd.DataFrame({"A":A,"region":region,"spending":spending,
                         "income":income,"debt":debt,"payhist":payhist,"default":observed})

PROXIES = ["region","spending"]
LEGIT   = ["income","debt","payhist"]
def features_for(cond):
    if cond == "with_protected":   return ["A"]+PROXIES+LEGIT
    if cond in ("drop_protected","gov_engine"): return PROXIES+LEGIT
    return LEGIT   # drop_proxies

def one_run(df, cond, seed):
    feats = features_for(cond)
    idx_tr, idx_te = train_test_split(np.arange(len(df)), test_size=0.3,
                                      random_state=seed, stratify=df["default"])
    g = df["A"].values[idx_te]; y = df["default"].values[idx_te]
    sc = StandardScaler().fit(df[feats].values[idx_tr])
    m = LogisticRegression(max_iter=1000).fit(sc.transform(df[feats].values[idx_tr]),
                                              df["default"].values[idx_tr])
    p = m.predict_proba(sc.transform(df[feats].values[idx_te]))[:,1]
    auc = roc_auc_score(y, p)
    if cond != "gov_engine":
        yp = (p>=0.5).astype(int)
    else:
        br = ((p>=0.5).astype(int)==0).mean()
        t1 = np.quantile(p[g==1], br); t0 = np.quantile(p[g==0], br)
        yp = np.where(g==1,(p>=t1),(p>=t0)).astype(int)
    dt, dfp = eo_gaps(y, yp, g)
    leak = leakage_auc(df[feats], df["A"].values) if cond in ("drop_protected","drop_proxies") else np.nan
    return auc, dir_ratio(yp,g), dt, dfp, leak

CONDS = ["with_protected","drop_protected","drop_proxies","gov_engine"]
SEEDS = list(range(30))
rows=[]
for cond in CONDS:
    arr = np.array([one_run(generate(s), cond, s) for s in SEEDS])
    m=np.nanmean(arr,0); ci=1.96*np.nanstd(arr,0)/np.sqrt(len(SEEDS))
    rows.append({"condition":cond,"AUC":m[0],"AUC_ci":ci[0],"DIR":m[1],"DIR_ci":ci[1],
                 "dTPR":m[2],"dFPR":m[3],"leak_AUC":m[4]})
res=pd.DataFrame(rows)
res.to_csv(os.path.join(TAB_DIR,"exp1_metrics.csv"),index=False)
print(res.round(3).to_string(index=False))
generate(42).to_csv(os.path.join(DATA_DIR,"synthetic_credit.csv"),index=False)

# Figure: AUC vs DIR scatter (seed 42)
df42=generate(42)
pts={c: one_run(df42,c,42)[:2] for c in CONDS}
markers=["o","s","^","D"]
fig,ax=plt.subplots(figsize=(6,4.2))
for i,c in enumerate(CONDS):
    d,a = pts[c][1], pts[c][0]
    ax.scatter(d,a,s=110,marker=markers[i],facecolor=GRAYS[i],edgecolor="black",
               linewidth=1.0,label=c,zorder=3)
ax.axvline(0.8,color="black",linestyle="--",linewidth=0.9,zorder=1)
ax.set_xlabel("Disparate Impact Ratio (DIR)"); ax.set_ylabel("AUC")
ax.set_xlim(0,1.15); ax.legend(frameon=True,edgecolor="black",loc="lower left")
fig.tight_layout(); save_fig(fig,"exp1_auc_dir"); plt.close(fig)

# Figure: DIR by condition with CI
pl=res.set_index("condition").loc[CONDS].reset_index()
fig,ax=plt.subplots(figsize=(6,4.2))
ax.bar(range(len(pl)),pl["DIR"],yerr=pl["DIR_ci"],capsize=4,edgecolor="black",
       linewidth=1.0,color=GRAYS)
ax.axhline(0.8,color="black",linestyle="--",linewidth=0.9)
ax.set_xticks(range(len(pl))); ax.set_xticklabels(pl["condition"],rotation=20,ha="right")
ax.set_ylabel("Disparate Impact Ratio (DIR)"); ax.set_ylim(0,1.15)
fig.tight_layout(); save_fig(fig,"exp1_dir_by_condition"); plt.close(fig)

print("Done: synthetic proxy-mechanism illustration, 30 seeds.")


     condition   AUC  AUC_ci   DIR  DIR_ci  dTPR  dFPR  leak_AUC
with_protected 0.872   0.003 0.737   0.012 0.038 0.180       NaN
drop_protected 0.865   0.003 0.871   0.009 0.106 0.081     0.919
  drop_proxies 0.860   0.003 1.004   0.009 0.230 0.011     0.501
    gov_engine 0.865   0.003 1.000   0.000 0.222 0.010       NaN


Done: synthetic proxy-mechanism illustration, 30 seeds.
